In [1]:
import os
import subprocess
import json

# Optional: if you want to force a specific GPU manually, set this to an integer like 1
FORCE_GPU_ID = None

# Minimum free memory (MiB) required to consider a GPU usable
MIN_FREE_MEM_MIB = 20000

# If True, prefer the GPU with maximum free memory among usable GPUs
PREFER_MAX_FREE_MEMORY = True

def query_gpus():
    cmd = [
        "nvidia-smi",
        "--query-gpu=index,memory.total,memory.used,memory.free,utilization.gpu",
        "--format=csv,noheader,nounits"
    ]
    out = subprocess.check_output(cmd, text=True).strip().splitlines()

    gpus = []
    for line in out:
        idx, total, used, free, util = [x.strip() for x in line.split(",")]
        gpus.append({
            "index": int(idx),
            "total_mib": int(total),
            "used_mib": int(used),
            "free_mib": int(free),
            "util_percent": int(util),
        })
    return gpus

def pick_best_gpu(gpus, min_free_mem_mib=20000):
    usable = [
        g for g in gpus
        if g["free_mib"] >= min_free_mem_mib
    ]

    if len(usable) == 0:
        raise RuntimeError(
            f"No GPU has at least {min_free_mem_mib} MiB free. "
            f"Available: {json.dumps(gpus, indent=2)}"
        )

    usable = sorted(
        usable,
        key=lambda x: (-x["free_mib"], x["util_percent"], x["used_mib"])
    )
    return usable[0]

gpus = query_gpus()

print("Visible GPU status:")
for g in gpus:
    print(
        f"GPU {g['index']}: free={g['free_mib']} MiB | "
        f"used={g['used_mib']} MiB / {g['total_mib']} MiB | "
        f"util={g['util_percent']}%"
    )

if FORCE_GPU_ID is not None:
    chosen_gpu = int(FORCE_GPU_ID)
    print(f"\nFORCE_GPU_ID is set. Using GPU {chosen_gpu}.")
else:
    best = pick_best_gpu(gpus, min_free_mem_mib=MIN_FREE_MEM_MIB)
    chosen_gpu = best["index"]
    print(f"\nAuto-selected GPU {chosen_gpu} with {best['free_mib']} MiB free.")

# Set BEFORE torch import
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = str(chosen_gpu)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("\nEnvironment set:")
print("CUDA_VISIBLE_DEVICES =", os.environ["CUDA_VISIBLE_DEVICES"])
print("PYTORCH_CUDA_ALLOC_CONF =", os.environ["PYTORCH_CUDA_ALLOC_CONF"])
print("\nIMPORTANT: If torch was already imported earlier in this kernel, restart the kernel and run from Cell 0 again.")


Visible GPU status:
GPU 0: free=1 MiB | used=143166 MiB / 143771 MiB | util=0%
GPU 1: free=141579 MiB | used=1588 MiB / 143771 MiB | util=0%
GPU 2: free=141579 MiB | used=1588 MiB / 143771 MiB | util=0%
GPU 3: free=141579 MiB | used=1588 MiB / 143771 MiB | util=0%
GPU 4: free=141579 MiB | used=1588 MiB / 143771 MiB | util=0%
GPU 5: free=141579 MiB | used=1588 MiB / 143771 MiB | util=0%
GPU 6: free=134117 MiB | used=9050 MiB / 143771 MiB | util=14%
GPU 7: free=141579 MiB | used=1588 MiB / 143771 MiB | util=0%

Auto-selected GPU 1 with 141579 MiB free.

Environment set:
CUDA_VISIBLE_DEVICES = 1
PYTORCH_CUDA_ALLOC_CONF = expandable_segments:True

IMPORTANT: If torch was already imported earlier in this kernel, restart the kernel and run from Cell 0 again.


In [2]:
import os
import gc
import json
import math
import copy
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForMaskedLM
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

try:
    from IPython.display import display
except Exception:
    display = print

In [ ]:
def clear_gpu_memory():
    """
    Clears the GPU cache and triggers Python garbage collection
    to free up memory for the next seed/task.
    """
    # 1. Clear out any unreferenced objects in Python
    gc.collect()
    
    # 2. Clear the PyTorch cache across all available GPUs
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        # If you want to be extra thorough on a multi-GPU DGX:
        torch.cuda.ipc_collect() 
        
    print(f"🧹 GPU Cache Cleared. Current Memory Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

In [4]:
import os
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))


CUDA_VISIBLE_DEVICES = 1


In [5]:
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

if torch.cuda.is_available():
    print("Visible CUDA device count:", torch.cuda.device_count())
    print("Current visible device index:", torch.cuda.current_device())
    print("Visible device 0 name:", torch.cuda.get_device_name(0))


DEVICE: cuda:0
Visible CUDA device count: 1
Current visible device index: 0
Visible device 0 name: NVIDIA H200


In [ ]:
GLOBAL_CONFIG = {
    # -------- runtime --------
    "seed": 13,
    "shots_per_class": 25,
    "n_subsets": 8,

    

    # -------- model --------
    "english_model_name": "bert-base-uncased",
    "chinese_model_name": "bert-base-chinese",
    "max_length": 256,
    "num_soft_tokens": 15,
    "dropout": 0.5,
    "freeze_plm": False,

    # -------- optimization --------
    "lr": 2e-5,
    "weight_decay": 1e-4,
    "l2_alpha": 1e-3,
    "batch_size": 32,              # DGX-friendly default; no 4x8 accumulation logic
    "accumulation_steps": 1,
    "source_epochs": 25,
    "refinement_epochs": 1,
    "final_epochs": 2,
    "grad_clip": 1.0,
    "amp": True,

    # -------- dataloader --------
    "num_workers": 0,
    "pin_memory": False,

    # -------- pseudo-label stabilization --------
    "train_refine_conf_threshold": 0.80,
    "final_invariant_conf_threshold": 0.90,
    "required_vote_ratio": 1.0,          # 1.0 = unanimous, keeps paper-style invariant filtering
    "use_source_replay_in_refinement": True,
    "source_replay_factor_refine": 4,
    "source_replay_factor_final": 10,
    "required_vote_ratio": 0.85, # Allows 6/7 votes to pass instead of 7/7

    # -------- prompt / verbalizer --------
    "label_map": {0: "Human", 1: "AI"},
    "prompt_text": {
        "english": "This text was written by",
        "chinese": "这段文本由",
    },
    "verbalizer": {
        "english": {
            0: ["human", "person", "people"],
            1: ["ai", "llm", "large language model"],
        },
        "chinese": {
            0: ["人类", "真人", "医生"],
            1: ["人工智能", "大语言模型", "AI"],
        },
    },

    # -------- save dirs --------
    "save_root": "./seedwise_paper_impl_outputs",
    "save_reports_dir": "./seedwise_paper_impl_outputs/reports",
    "save_votes_dir": "./seedwise_paper_impl_outputs/votes",
    "save_histories_dir": "./seedwise_paper_impl_outputs/histories",
    "save_checkpoints_dir": "./seedwise_paper_impl_outputs/checkpoints",
    "save_configs_dir": "./seedwise_paper_impl_outputs/configs",
}

for k in [
    "save_root",
    "save_reports_dir",
    "save_votes_dir",
    "save_histories_dir",
    "save_checkpoints_dir",
    "save_configs_dir",
]:
    os.makedirs(GLOBAL_CONFIG[k], exist_ok=True)

In [ ]:
CHEAT_ROOT = "/nfsshare/users/P127003093/Mini Project/datasets/CHEAT"
MEDQA_ROOT = r"datasets\Chinese Medical QnA"

In [8]:
TASK_REGISTRY = {
    # -------- CHEAT --------
    "A->M": {
        "source_file": os.path.join(CHEAT_ROOT, "Algorithm.csv"),
        "target_file": os.path.join(CHEAT_ROOT, "Medicine.csv"),
        "dataset_type": "english",
    },
    "A->S": {
        "source_file": os.path.join(CHEAT_ROOT, "Algorithm.csv"),
        "target_file": os.path.join(CHEAT_ROOT, "Subject.csv"),
        "dataset_type": "english",
    },
    "M->A": {
        "source_file": os.path.join(CHEAT_ROOT, "Medicine.csv"),
        "target_file": os.path.join(CHEAT_ROOT, "Algorithm.csv"),
        "dataset_type": "english",
    },
    "M->S": {
        "source_file": os.path.join(CHEAT_ROOT, "Medicine.csv"),
        "target_file": os.path.join(CHEAT_ROOT, "Subject.csv"),
        "dataset_type": "english",
    },
    "S->A": {
        "source_file": os.path.join(CHEAT_ROOT, "Subject.csv"),
        "target_file": os.path.join(CHEAT_ROOT, "Algorithm.csv"),
        "dataset_type": "english",
    },
    "S->M": {
        "source_file": os.path.join(CHEAT_ROOT, "Subject.csv"),
        "target_file": os.path.join(CHEAT_ROOT, "Medicine.csv"),
        "dataset_type": "english",
    },

    # -------- Chinese Medical QA --------
    "I->O": {
        "source_file": os.path.join(MEDQA_ROOT, "IM_2000_AI_HUMAN.csv"),
        "target_file": os.path.join(MEDQA_ROOT, "Oncology_2000_AI_HUMAN.csv"),
        "dataset_type": "chinese",
    },
    "I->P": {
        "source_file": os.path.join(MEDQA_ROOT, "IM_2000_AI_HUMAN.csv"),
        "target_file": os.path.join(MEDQA_ROOT, "Pediatrics_2000_AI_HUMAN.csv"),
        "dataset_type": "chinese",
    },
    "O->I": {
        "source_file": os.path.join(MEDQA_ROOT, "Oncology_2000_AI_HUMAN.csv"),
        "target_file": os.path.join(MEDQA_ROOT, "IM_2000_AI_HUMAN.csv"),
        "dataset_type": "chinese",
    },
    "O->P": {
        "source_file": os.path.join(MEDQA_ROOT, "Oncology_2000_AI_HUMAN.csv"),
        "target_file": os.path.join(MEDQA_ROOT, "Pediatrics_2000_AI_HUMAN.csv"),
        "dataset_type": "chinese",
    },
    "P->I": {
        "source_file": os.path.join(MEDQA_ROOT, "Pediatrics_2000_AI_HUMAN.csv"),
        "target_file": os.path.join(MEDQA_ROOT, "IM_2000_AI_HUMAN.csv"),
        "dataset_type": "chinese",
    },
    "P->O": {
        "source_file": os.path.join(MEDQA_ROOT, "Pediatrics_2000_AI_HUMAN.csv"),
        "target_file": os.path.join(MEDQA_ROOT, "Oncology_2000_AI_HUMAN.csv"),
        "dataset_type": "chinese",
    },
}

In [9]:
print("Registered tasks:")
for k, v in TASK_REGISTRY.items():
    print(k, "->", v["dataset_type"], "|", v["source_file"], "=>", v["target_file"])

Registered tasks:
A->M -> english | /nfsshare/users/P127003093/Mini Project/datasets/CHEAT/Algorithm.csv => /nfsshare/users/P127003093/Mini Project/datasets/CHEAT/Medicine.csv
A->S -> english | /nfsshare/users/P127003093/Mini Project/datasets/CHEAT/Algorithm.csv => /nfsshare/users/P127003093/Mini Project/datasets/CHEAT/Subject.csv
M->A -> english | /nfsshare/users/P127003093/Mini Project/datasets/CHEAT/Medicine.csv => /nfsshare/users/P127003093/Mini Project/datasets/CHEAT/Algorithm.csv
M->S -> english | /nfsshare/users/P127003093/Mini Project/datasets/CHEAT/Medicine.csv => /nfsshare/users/P127003093/Mini Project/datasets/CHEAT/Subject.csv
S->A -> english | /nfsshare/users/P127003093/Mini Project/datasets/CHEAT/Subject.csv => /nfsshare/users/P127003093/Mini Project/datasets/CHEAT/Algorithm.csv
S->M -> english | /nfsshare/users/P127003093/Mini Project/datasets/CHEAT/Subject.csv => /nfsshare/users/P127003093/Mini Project/datasets/CHEAT/Medicine.csv
I->O -> chinese | datasets\Chinese Medic

In [10]:
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [11]:
def gpu_mem_text():
    if not torch.cuda.is_available():
        return "cpu"
    allocated = torch.cuda.memory_allocated() / (1024 ** 3)
    reserved = torch.cuda.memory_reserved() / (1024 ** 3)
    return f"{allocated:.2f}G/{reserved:.2f}G"

In [12]:
def stage_banner(task_name: str, stage_name: str, extra: str = ""):
    print("\n" + "=" * 120)
    print(f"[TASK={task_name}] [{stage_name}] DEVICE={DEVICE} GPU_MEM={gpu_mem_text()} {extra}")
    print("=" * 120)

In [13]:
def task_prefix(task_name: str, seed: int) -> str:
    return f"{task_name.replace('->', '_')}_{seed}"

In [14]:

def save_csv(df: pd.DataFrame, path: str):
    df.to_csv(path, index=False)
    return path

In [15]:
def save_json(obj: dict, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    return path


In [ ]:
def clean_text(x: str) -> str:
    return " ".join(str(x).split()).strip()


In [ ]:
def load_domain_csv(filename: str) -> pd.DataFrame:
    df = pd.read_csv(filename).copy()
    return df.reset_index(drop=True)


In [18]:
def infer_model_name(dataset_type: str) -> str:
    if dataset_type == "english":
        return GLOBAL_CONFIG["english_model_name"]
    elif dataset_type == "chinese":
        return GLOBAL_CONFIG["chinese_model_name"]
    raise ValueError(f"Unsupported dataset_type={dataset_type}")

In [ ]:
def build_model_text(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    cols = set(df.columns)

    if {"title", "keyword", "abstract"}.issubset(cols):
        df["model_text"] = (
            "Title: " + df["title"].fillna("").astype(str).map(clean_text) + " "
            "Keyword: " + df["keyword"].fillna("").astype(str).map(clean_text) + " "
            "Abstract: " + df["abstract"].fillna("").astype(str).map(clean_text)
        )
    elif {"dept", "title", "question", "answer"}.issubset(cols):
        df["model_text"] = (
            "Department: " + df["dept"].fillna("").astype(str).map(clean_text) + " "
            "Title: " + df["title"].fillna("").astype(str).map(clean_text) + " "
            "Question: " + df["question"].fillna("").astype(str).map(clean_text) + " "
            "Answer: " + df["answer"].fillna("").astype(str).map(clean_text)
        )
    elif "abstract" in cols:
        df["model_text"] = df["abstract"].fillna("").astype(str).map(clean_text)
    elif "answer" in cols:
        df["model_text"] = df["answer"].fillna("").astype(str).map(clean_text)
    else:
        raise ValueError(f"Unsupported columns: {df.columns.tolist()}")

    if "label" not in df.columns:
        raise ValueError("CSV must contain a 'label' column.")

    df["label"] = df["label"].astype(int)
    label_values = sorted(df["label"].dropna().unique().tolist())
    if not set(label_values).issubset({0, 1}):
        raise ValueError(f"Labels must be subset of {{0,1}}. Found: {label_values}")

    return df.reset_index(drop=True)

In [20]:
def sample_few_shot_per_class(df: pd.DataFrame, shots_per_class: int, seed: int) -> pd.DataFrame:
    pieces = []
    for label in sorted(df["label"].unique()):
        part = df[df["label"] == label].sample(n=shots_per_class, random_state=seed)
        pieces.append(part)
    out = pd.concat(pieces, axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

In [21]:
def inspect_task_data(task_name: str, shots_per_class: int, seed: int):
    spec = TASK_REGISTRY[task_name]
    source_df = build_model_text(load_domain_csv(spec["source_file"]))
    target_df = build_model_text(load_domain_csv(spec["target_file"]))

    print(f"\nTask: {task_name}")
    print("Dataset type:", spec["dataset_type"])
    print("Source shape:", source_df.shape)
    print("Target shape:", target_df.shape)
    print("Source labels:\n", source_df["label"].value_counts().sort_index())
    print("Target labels:\n", target_df["label"].value_counts().sort_index())

    few_df = sample_few_shot_per_class(source_df, shots_per_class, seed)
    print("Few-shot source shape:", few_df.shape)
    print("Few-shot label counts:\n", few_df["label"].value_counts().sort_index())

    display(source_df.head(2))
    display(target_df.head(2))

In [22]:
def load_task_data(task_name: str, shots_per_class: int, seed: int):
    spec = TASK_REGISTRY[task_name]
    source_df = build_model_text(load_domain_csv(spec["source_file"]))
    target_df = build_model_text(load_domain_csv(spec["target_file"]))
    source_few_df = sample_few_shot_per_class(source_df, shots_per_class, seed)
    target_df = target_df.reset_index(drop=True).copy()
    target_df["gold_label"] = target_df["label"].astype(int)
    return source_few_df, target_df, spec["dataset_type"]

In [23]:
def resolve_phrase_verbalizer(tokenizer, verbalizer: Dict[int, List[str]]) -> Dict[int, List[List[int]]]:
    resolved = {}
    for label, phrases in verbalizer.items():
        ids_list = []
        for phrase in phrases:
            ids = tokenizer.encode(phrase, add_special_tokens=False)
            if len(ids) == 0:
                raise ValueError(f"Phrase tokenized to empty sequence: {phrase}")
            ids_list.append(ids)
        resolved[label] = ids_list
    return resolved

In [24]:
@dataclass
class MultiMaskSoftPromptCollator:
    tokenizer: AutoTokenizer
    max_length: int
    num_label_masks: int
    prompt_text: str

    def __call__(self, batch):
        input_ids_list = []
        attention_mask_list = []
        mask_positions_list = []
        labels = []
        idxs = []

        prompt_ids = self.tokenizer.encode(self.prompt_text, add_special_tokens=False)
        period_ids = self.tokenizer.encode(".", add_special_tokens=False)
        cls_id = self.tokenizer.cls_token_id
        sep_id = self.tokenizer.sep_token_id
        pad_id = self.tokenizer.pad_token_id
        mask_id = self.tokenizer.mask_token_id

        reserved = 1 + len(prompt_ids) + self.num_label_masks + len(period_ids) + 1
        text_budget = self.max_length - reserved
        if text_budget <= 0:
            raise ValueError("max_length is too small for prompt construction.")

        for item in batch:
            text = clean_text(item["text"])
            text_ids = self.tokenizer.encode(
                text,
                add_special_tokens=False,
                truncation=True,
                max_length=text_budget,
            )

            input_ids = [cls_id] + text_ids + prompt_ids + ([mask_id] * self.num_label_masks) + period_ids + [sep_id]
            mask_positions = [i for i, x in enumerate(input_ids) if x == mask_id]
            if len(mask_positions) != self.num_label_masks:
                raise ValueError("Mask positions could not be constructed correctly.")

            attention_mask = [1] * len(input_ids)
            pad_len = self.max_length - len(input_ids)
            input_ids += [pad_id] * pad_len
            attention_mask += [0] * pad_len

            input_ids_list.append(input_ids)
            attention_mask_list.append(attention_mask)
            mask_positions_list.append(mask_positions)
            idxs.append(item["idx"])
            if "label" in item:
                labels.append(int(item["label"]))

        out = {
            "input_ids": torch.tensor(input_ids_list, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask_list, dtype=torch.long),
            "mask_positions": torch.tensor(mask_positions_list, dtype=torch.long),
            "idxs": torch.tensor(idxs, dtype=torch.long),
        }
        if len(labels) > 0:
            out["labels"] = torch.tensor(labels, dtype=torch.long)
        return out

In [25]:
def build_tokenizer_and_collator(dataset_type: str):
    model_name = infer_model_name(dataset_type)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    verbalizer = GLOBAL_CONFIG["verbalizer"][dataset_type]
    resolved_verbalizer = resolve_phrase_verbalizer(tokenizer, verbalizer)
    num_label_masks = max(len(ids) for phrases in resolved_verbalizer.values() for ids in phrases)
    collator = MultiMaskSoftPromptCollator(
        tokenizer=tokenizer,
        max_length=GLOBAL_CONFIG["max_length"],
        num_label_masks=num_label_masks,
        prompt_text=GLOBAL_CONFIG["prompt_text"][dataset_type],
    )
    return tokenizer, resolved_verbalizer, num_label_masks, collator


In [26]:
class TextFrameDataset(Dataset):
    def __init__(self, df: pd.DataFrame, text_col: str = "model_text", label_col: str = "label", labeled: bool = True):
        self.df = df.reset_index(drop=True).copy()
        self.texts = self.df[text_col].fillna("").astype(str).tolist()
        self.labeled = labeled
        self.labels = self.df[label_col].astype(int).tolist() if labeled else None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = {"text": self.texts[idx], "idx": idx}
        if self.labeled:
            item["label"] = self.labels[idx]
        return item

In [27]:
class PaperSoftPromptDetector(nn.Module):
    def __init__(
        self,
        model_name: str,
        tokenizer,
        resolved_verbalizer: Dict[int, List[List[int]]],
        num_soft_tokens: int = 10,
        dropout: float = 0.5,
        freeze_plm: bool = False,
    ):
        super().__init__()
        self.tokenizer = tokenizer
        self.resolved_verbalizer = resolved_verbalizer
        self.num_soft_tokens = num_soft_tokens

        self.mlm = AutoModelForMaskedLM.from_pretrained(model_name)
        self.hidden_size = self.mlm.config.hidden_size
        self.soft_prompt = nn.Parameter(torch.randn(num_soft_tokens, self.hidden_size) * 0.02)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer("soft_prompt_attention", torch.ones(num_soft_tokens, dtype=torch.long))

        if freeze_plm:
            for p in self.mlm.parameters():
                p.requires_grad = False

    def _get_backbone(self):
        if hasattr(self.mlm, "bert"):
            return self.mlm.bert
        raise ValueError("This implementation expects a BERT-style masked LM backbone.")

    def _get_mask_logits(self, vocab_logits, mask_positions):
        batch_size, num_masks = mask_positions.shape
        batch_idx = torch.arange(batch_size, device=vocab_logits.device).unsqueeze(1).expand(batch_size, num_masks)
        return vocab_logits[batch_idx, mask_positions, :]

    def _phrase_probability(self, mask_probs, token_ids: List[int]):
        phrase_len = len(token_ids)
        token_ids = torch.tensor(token_ids, device=mask_probs.device, dtype=torch.long)

        selected_probs = mask_probs[:, :phrase_len, :]
        token_probs = selected_probs.gather(
            dim=-1,
            index=token_ids.unsqueeze(0).unsqueeze(-1).expand(selected_probs.size(0), -1, 1)
        ).squeeze(-1)
        phrase_prob = token_probs.mean(dim=-1)
        return phrase_prob

    def _class_logits_from_verbalizer(self, mask_logits):
        mask_probs = F.softmax(mask_logits, dim=-1)
        class_scores = []

        for label in [0, 1]:
            phrase_scores = []
            for token_ids in self.resolved_verbalizer[label]:
                phrase_scores.append(self._phrase_probability(mask_probs, token_ids))
            phrase_scores = torch.stack(phrase_scores, dim=-1)
            class_score = phrase_scores.mean(dim=-1)
            class_scores.append(class_score)

        class_scores = torch.stack(class_scores, dim=-1)
        class_logits = torch.log(class_scores + 1e-12)
        return class_logits

    def forward(self, input_ids, attention_mask, mask_positions, labels=None):
        batch_size = input_ids.size(0)

        token_embeds = self.mlm.get_input_embeddings()(input_ids)
        soft_prompt = self.soft_prompt.unsqueeze(0).expand(batch_size, -1, -1)

        cls_embed = token_embeds[:, 0:1, :]
        rest_embeds = token_embeds[:, 1:, :]
        inputs_embeds = torch.cat([cls_embed, soft_prompt, rest_embeds], dim=1)

        cls_mask = attention_mask[:, 0:1]
        rest_mask = attention_mask[:, 1:]
        prompt_mask = self.soft_prompt_attention.unsqueeze(0).expand(batch_size, -1)
        ext_attention_mask = torch.cat([cls_mask, prompt_mask, rest_mask], dim=1)

        ext_mask_positions = mask_positions + self.num_soft_tokens

        backbone = self._get_backbone()
        outputs = backbone(
            inputs_embeds=inputs_embeds,
            attention_mask=ext_attention_mask,
            return_dict=True,
        )

        hidden = self.dropout(outputs.last_hidden_state)
        vocab_logits = self.mlm.cls(hidden)
        mask_logits = self._get_mask_logits(vocab_logits, ext_mask_positions)
        class_logits = self._class_logits_from_verbalizer(mask_logits)

        out = {
            "class_logits": class_logits,
            "mask_logits": mask_logits,
        }
        if labels is not None:
            out["ce_loss"] = F.cross_entropy(class_logits, labels)
        return out

In [28]:
def build_model_objects(dataset_type: str):
    tokenizer, resolved_verbalizer, num_label_masks, collator = build_tokenizer_and_collator(dataset_type)
    model = PaperSoftPromptDetector(
        model_name=infer_model_name(dataset_type),
        tokenizer=tokenizer,
        resolved_verbalizer=resolved_verbalizer,
        num_soft_tokens=GLOBAL_CONFIG["num_soft_tokens"],
        dropout=GLOBAL_CONFIG["dropout"],
        freeze_plm=GLOBAL_CONFIG["freeze_plm"],
    ).to(DEVICE)
    return tokenizer, resolved_verbalizer, num_label_masks, collator, model


In [ ]:
def make_loader(df: pd.DataFrame, collator, batch_size: int, shuffle: bool, labeled: bool):
    dataset = TextFrameDataset(df, labeled=labeled)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=GLOBAL_CONFIG["num_workers"],
        pin_memory=GLOBAL_CONFIG["pin_memory"],
        collate_fn=collator,
    )

In [30]:
def l2_regularization(model: nn.Module):
    reg = torch.tensor(0.0, device=DEVICE)
    for p in model.parameters():
        if p.requires_grad:
            reg = reg + torch.sum(p ** 2)
    return reg

In [ ]:
@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss, total_count = 0.0, 0

    autocast_enabled = torch.cuda.is_available() and GLOBAL_CONFIG["amp"]
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.amp.autocast(device_type="cuda", enabled=autocast_enabled):
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                mask_positions=batch["mask_positions"],
                labels=batch["labels"],
            )
        preds = torch.argmax(outputs["class_logits"], dim=-1)
        bs = batch["labels"].size(0)

        total_loss += outputs["ce_loss"].item() * bs
        total_count += bs
        all_preds.extend(preds.detach().cpu().tolist())
        all_labels.extend(batch["labels"].detach().cpu().tolist())

    return {
        "loss": total_loss / max(total_count, 1),
        "acc": accuracy_score(all_labels, all_preds),
        "f1": f1_score(all_labels, all_preds, zero_division=0),
    }

In [32]:
@torch.no_grad()
def predict_target_labels(model, df, collator, batch_size, desc="Predicting", temperature=1.5):
    model.eval()
    loader = make_loader(df, collator, batch_size=batch_size, shuffle=False, labeled=False)

    all_preds = []
    all_probs = []

    autocast_enabled = torch.cuda.is_available() and GLOBAL_CONFIG["amp"]
    pbar = tqdm(loader, desc=desc, leave=False)
    for batch in pbar:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.amp.autocast(device_type="cuda", enabled=autocast_enabled):
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                mask_positions=batch["mask_positions"],
            )
            
        # ---> FIX: Temperature Scaling (Smoothing Overconfidence) <---
        scaled_logits = outputs["class_logits"] / temperature
        probs = torch.softmax(scaled_logits, dim=-1)
        max_probs, preds = torch.max(probs, dim=-1)

        all_preds.extend(preds.detach().cpu().tolist())
        all_probs.extend(max_probs.detach().cpu().tolist())
        pbar.set_postfix(mem=gpu_mem_text())

    return np.array(all_preds, dtype=int), np.array(all_probs, dtype=float)

In [33]:
# =========================================
# REPLACEMENT FOR CELL 33: train_model
# =========================================
def train_model(
    model, train_loader, epochs: int, lr: float, weight_decay: float, l2_alpha: float,
    task_name: str, stage_name: str, seed: int, iteration: int,
):
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr, weight_decay=weight_decay,
    )

    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available() and GLOBAL_CONFIG["amp"])
    accumulation_steps = max(1, GLOBAL_CONFIG["accumulation_steps"])
    grad_clip = GLOBAL_CONFIG["grad_clip"]

    history = []
    model.to(DEVICE)

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss, running_examples, running_correct = 0.0, 0, 0
        optimizer.zero_grad(set_to_none=True)
        
        desc = f"{stage_name} | task={task_name} | seed={seed} | iter={iteration} | epoch={epoch}/{epochs}"
        pbar = tqdm(train_loader, desc=desc, leave=False)

        for step_idx, batch in enumerate(pbar, start=1):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            autocast_enabled = torch.cuda.is_available() and GLOBAL_CONFIG["amp"]

            with torch.amp.autocast(device_type="cuda", enabled=autocast_enabled):
                outputs = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    mask_positions=batch["mask_positions"],
                    labels=batch["labels"],
                )
                
                class_logits = outputs["class_logits"]
                labels = batch["labels"]
                
                # Base Paper Eq 6: Pure Cross Entropy + L2 Regularization
                if iteration == 0 or iteration == 9:
                    # Clean data: train on everything
                    ce_loss = F.cross_entropy(class_logits, labels)
                else:
                    # Brainstorm Defense: Confidence Masking (Ignore noisy pseudo-labels)
                    probs = F.softmax(class_logits, dim=-1)
                    conf = probs.gather(1, labels.unsqueeze(1)).squeeze(1)
                    
                    # Only calculate loss if confidence is above threshold (e.g. 0.80)
                    conf_thresh = GLOBAL_CONFIG.get("train_refine_conf_threshold", 0.80)
                    mask = (conf >= conf_thresh).float()
                    
                    unreduced_loss = F.cross_entropy(class_logits, labels, reduction='none')
                    # Mean loss only over the confident samples to prevent error propagation
                    ce_loss = (unreduced_loss * mask).sum() / max(mask.sum(), 1.0)

                # Eq 7: Apply the paper's L2 Regularization constraint to prevent overfitting
                reg = l2_regularization(model)
                loss = (ce_loss + l2_alpha * reg) / accumulation_steps

            scaler.scale(loss).backward()

            did_step = 0
            if (step_idx % accumulation_steps == 0) or (step_idx == len(train_loader)):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                did_step = 1

            preds = torch.argmax(class_logits, dim=-1)
            bs = labels.size(0)

            running_loss += (loss.item() * accumulation_steps) * bs
            running_examples += bs
            running_correct += (preds == labels).sum().item()

            pbar.set_postfix(
                run_acc=f"{running_correct / max(running_examples, 1):.4f}",
                run_loss=f"{running_loss / max(running_examples, 1):.4f}",
                opt_step=did_step, mem=gpu_mem_text(),
            )

        epoch_row = {
            "task": task_name, "seed": seed, "stage": stage_name,
            "iteration": iteration, "epoch": epoch,
            "train_loss": running_loss / max(running_examples, 1),
            "train_acc": running_correct / max(running_examples, 1),
        }
        history.append(epoch_row)

        print(f"[{stage_name}] iter={iteration} epoch={epoch}/{epochs} train_loss={epoch_row['train_loss']:.6f} train_acc={epoch_row['train_acc']:.4f}")

    return model, pd.DataFrame(history)

In [34]:
def evaluate_full_target(model, target_df, collator):
    loader = make_loader(target_df[["model_text", "label"]].copy(), collator, GLOBAL_CONFIG["batch_size"], False, True)
    return evaluate_model(model, loader)


In [35]:
def split_indices_evenly(n: int, n_subsets: int, seed: int) -> List[np.ndarray]:
    rng = np.random.RandomState(seed)
    shuffled = rng.permutation(n)
    splits = np.array_split(shuffled, n_subsets)
    return [np.array(s, dtype=int) for s in splits]

In [36]:
def majority_vote_with_confidence(votes: List[List[int]], probs: List[List[float]]):
    final_labels = []
    final_confidences = []
    vote_ratios = []

    for vote_list, prob_list in zip(votes, probs):
        counts = np.bincount(np.array(vote_list, dtype=int), minlength=2)

        if counts[0] == counts[1]:
            mean0 = np.mean([p for v, p in zip(vote_list, prob_list) if v == 0]) if 0 in vote_list else 0.0
            mean1 = np.mean([p for v, p in zip(vote_list, prob_list) if v == 1]) if 1 in vote_list else 0.0
            winner = 1 if mean1 > mean0 else 0
        else:
            winner = int(np.argmax(counts))

        winner_probs = [p for v, p in zip(vote_list, prob_list) if v == winner]
        winner_conf = float(np.mean(winner_probs)) if len(winner_probs) > 0 else 0.0
        vote_ratio = float(counts[winner] / len(vote_list))

        final_labels.append(winner)
        final_confidences.append(winner_conf)
        vote_ratios.append(vote_ratio)

    return (
        np.array(final_labels, dtype=int),
        np.array(final_confidences, dtype=float),
        np.array(vote_ratios, dtype=float),
    )

In [37]:
def attach_vote_columns(df: pd.DataFrame, votes: List[List[int]], probs: List[List[float]]) -> pd.DataFrame:
    df = df.copy()
    df["vote_history_label"] = [json.dumps(v) for v in votes]
    df["vote_history_prob"] = [json.dumps([round(float(x), 6) for x in p]) for p in probs]
    df["vote_count"] = [len(v) for v in votes]
    return df


In [ ]:
def build_refinement_train_df(
    initial_pseudo_df: pd.DataFrame,
    subset_indices: np.ndarray,
    source_df: pd.DataFrame,
    seed: int
):
    subset_df = initial_pseudo_df.iloc[subset_indices].copy().reset_index(drop=True)

    # 1. Filter out noisy predictions using confidence threshold
    conf_thresh = GLOBAL_CONFIG.get("train_refine_conf_threshold", 0.0)
    if conf_thresh > 0:
        confident_mask = subset_df["pseudo_conf"] >= conf_thresh
        subset_df = subset_df[confident_mask].copy()

    train_df = (
        subset_df[["model_text", "pseudo_label"]]
        .rename(columns={"pseudo_label": "label"})
        .copy()
    )

    # 2. SOURCE REPLAY: Mix high-quality source data to prevent catastrophic forgetting
    if GLOBAL_CONFIG.get("use_source_replay_in_refinement", True):
        factor = GLOBAL_CONFIG.get("source_replay_factor_refine", 4)
        source_repeated = pd.concat([source_df[["model_text", "label"]]] * factor, ignore_index=True)
        train_df = pd.concat([train_df, source_repeated], ignore_index=True)

    # Shuffle the combined dataset
    train_df = train_df.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    info = {
        "subset_size": len(subset_indices),
        "confident_subset_size": len(subset_df),
        "refine_train_size": len(train_df),
    }
    return train_df, info

In [39]:
def select_invariant_samples(voted_target_df: pd.DataFrame):
    df = voted_target_df.copy().reset_index(drop=True)
    required_votes = GLOBAL_CONFIG["n_subsets"] - 1

    keep_rows = []
    pseudo_labels = []
    pseudo_confs = []
    vote_ratios = []

    # Fetch configuration or use your specified defaults
    req_ratio = GLOBAL_CONFIG.get("required_vote_ratio", 0.85) 
    final_conf_thresh = GLOBAL_CONFIG.get("final_invariant_conf_threshold", 0.90)

    for i in range(len(df)):
        vote_list = df.loc[i, "vote_history_label"]
        prob_list = df.loc[i, "vote_history_prob"]

        if isinstance(vote_list, str):
            vote_list = json.loads(vote_list)
        if isinstance(prob_list, str):
            prob_list = json.loads(prob_list)

        vote_list = [int(x) for x in vote_list]
        prob_list = [float(x) for x in prob_list]

        # A sample must have been predicted in all held-out rounds
        if len(vote_list) != required_votes:
            continue

        # Calculate vote counts and identify the majority winner
        counts = np.bincount(vote_list, minlength=2)
        winner = int(np.argmax(counts))
        ratio = counts[winner] / len(vote_list)

        # Calculate the confidence ONLY for the votes that agreed with the winner
        winner_probs = [p for v, p in zip(vote_list, prob_list) if v == winner]
        conf = float(np.mean(winner_probs)) if len(winner_probs) > 0 else 0.0

        # Keep if it meets the relaxed consensus and high confidence threshold
        if ratio >= req_ratio and conf >= final_conf_thresh:
            keep_rows.append(i)
            pseudo_labels.append(winner)
            pseudo_confs.append(conf)
            vote_ratios.append(ratio)

    invariant_df = df.iloc[keep_rows].copy().reset_index(drop=True)
    invariant_df["pseudo_label"] = np.array(pseudo_labels, dtype=int) if len(pseudo_labels) > 0 else []
    invariant_df["pseudo_conf"] = np.array(pseudo_confs, dtype=float) if len(pseudo_confs) > 0 else []
    invariant_df["vote_ratio"] = np.array(vote_ratios, dtype=float) if len(vote_ratios) > 0 else []

    stats = {
        "total_target_samples": len(df),
        "invariant_samples": len(invariant_df),
        "retention_rate": len(invariant_df) / max(len(df), 1),
        "invariant_pseudo_acc": (
            accuracy_score(invariant_df["gold_label"], invariant_df["pseudo_label"])
            if len(invariant_df) > 0 else np.nan
        ),
        "invariant_pseudo_f1": (
            f1_score(invariant_df["gold_label"], invariant_df["pseudo_label"], zero_division=0)
            if len(invariant_df) > 0 else np.nan
        ),
    }
    return invariant_df, stats

In [40]:
def run_source_training(task_name: str, seed: int, shots_per_class: int):
    stage_banner(task_name, "SOURCE TRAINING", f"seed={seed} shots={shots_per_class}")

    seed_everything(seed)
    source_df, target_df, dataset_type = load_task_data(task_name, shots_per_class, seed)
    tokenizer, resolved_verbalizer, num_label_masks, collator, model = build_model_objects(dataset_type)

    source_loader = make_loader(source_df[["model_text", "label"]], collator, GLOBAL_CONFIG["batch_size"], True, True)

    model, source_hist = train_model(
        model=model,
        train_loader=source_loader,
        epochs=GLOBAL_CONFIG["source_epochs"],
        lr=GLOBAL_CONFIG["lr"],
        weight_decay=GLOBAL_CONFIG["weight_decay"],
        l2_alpha=GLOBAL_CONFIG["l2_alpha"],
        task_name=task_name,
        stage_name="SOURCE_TRAIN",
        seed=seed,
        iteration=0,
    )

    target_metrics = evaluate_full_target(model, target_df, collator)
    report_rows = [{
        "task": task_name,
        "seed": seed,
        "shots_per_class": shots_per_class,
        "n_subsets": GLOBAL_CONFIG["n_subsets"],
        "stage": "iter_0",
        "iteration": 0,
        "target_loss": target_metrics["loss"],
        "target_acc": target_metrics["acc"],
        "target_f1": target_metrics["f1"],
        "subset_size": np.nan,
        "confident_subset_size": np.nan,
        "refine_train_size": np.nan,
        "invariant_samples": np.nan,
        "retention_rate": np.nan,
    }]

    return {
        "source_df": source_df,
        "target_df": target_df,
        "dataset_type": dataset_type,
        "collator": collator,
        "model": model,
        "source_history_df": source_hist,
        "report_rows": report_rows,
    }

In [41]:
# =========================================
# REPLACEMENT FOR CELL 41: run_iterative_refinement
# =========================================
def run_iterative_refinement(task_name: str, seed: int, source_df: pd.DataFrame, target_df: pd.DataFrame, collator, model):
    stage_banner(task_name, "ITERATIVE REFINEMENT", f"seed={seed} n_subsets={GLOBAL_CONFIG['n_subsets']}")

    df = target_df.copy().reset_index(drop=True)

    # ---> STEP 0: Establish Unsupervised Baseline <---
    init_preds, init_probs = predict_target_labels(
        model, df, collator, GLOBAL_CONFIG["batch_size"],
        desc=f"INIT_PSEUDO | {task_name} | seed={seed}", temperature=1.5
    )

    best_mean_conf = float(np.mean(init_probs))
    print(f"--- [Unsupervised Baseline] Iteration 0 Mean Conf: {best_mean_conf:.4f} ---")

    df["init_pseudo_label"] = init_preds.astype(int)
    df["init_pseudo_conf"] = init_probs.astype(float)

    initial_pseudo_df = df[["model_text", "gold_label", "init_pseudo_label", "init_pseudo_conf"]].copy()
    initial_pseudo_df = initial_pseudo_df.rename(columns={"init_pseudo_label": "pseudo_label", "init_pseudo_conf": "pseudo_conf"})
    dynamic_pseudo_df = initial_pseudo_df.copy()

    subset_indices_list = split_indices_evenly(len(df), GLOBAL_CONFIG["n_subsets"], seed)
    all_indices = np.arange(len(df))

    votes = [[] for _ in range(len(df))]
    probs = [[] for _ in range(len(df))]

    iter_rows = []
    refine_histories = []

    for iter_id, train_indices in enumerate(subset_indices_list, start=1):
        print(f"\n--- [{task_name}] Refinement iteration {iter_id}/{GLOBAL_CONFIG['n_subsets']} ---")

        refine_train_df, info = build_refinement_train_df(dynamic_pseudo_df, train_indices, source_df, seed)
        refine_loader = make_loader(refine_train_df, collator, GLOBAL_CONFIG["batch_size"], True, True)

        # Train the model (updates weights in-place, unfrozen PLM)
        model, hist = train_model(
            model=model, train_loader=refine_loader, epochs=GLOBAL_CONFIG["refinement_epochs"],
            lr=GLOBAL_CONFIG["lr"], weight_decay=GLOBAL_CONFIG["weight_decay"], l2_alpha=GLOBAL_CONFIG["l2_alpha"],
            task_name=task_name, stage_name="REFINE_TRAIN", seed=seed, iteration=iter_id,
        )

        if len(hist) > 0:
            hist = hist.copy()
            hist["subset_size"] = info["subset_size"]
            hist["confident_subset_size"] = info["confident_subset_size"]
            hist["refine_train_size"] = info["refine_train_size"]
            refine_histories.append(hist)

        heldout_indices = np.setdiff1d(all_indices, train_indices)
        heldout_df = df.iloc[heldout_indices].copy().reset_index(drop=True)

        heldout_preds, heldout_probs = predict_target_labels(
            model, heldout_df, collator, GLOBAL_CONFIG["batch_size"],
            desc=f"REFINE_PREDICT | {task_name} | seed={seed} | iter={iter_id}", temperature=1.5
        )

        current_mean_conf = float(np.mean(heldout_probs))
        
        # 1. Anti-Collapse Check (Quarantine)
        pred_ratio = float(np.mean(heldout_preds))
        is_collapsed = (pred_ratio < 0.10) or (pred_ratio > 0.90)
        
        # 2. Stability Check (Quarantine)
        prev_labels = dynamic_pseudo_df.loc[heldout_indices, "pseudo_label"].values
        stability = float(np.mean(heldout_preds == prev_labels))
        is_unstable = stability < 0.60
        
        # 3. The Unsupervised Gate
        is_better = (current_mean_conf > best_mean_conf) and (not is_collapsed) and (not is_unstable)

        for local_i, global_i in enumerate(heldout_indices):
            # Always track votes for Phase 3 (Voting Mechanism)
            votes[global_i].append(int(heldout_preds[local_i]))
            probs[global_i].append(float(heldout_probs[local_i]))
            
            # ---> Brainstorm Defense: Isolate the noise. Only update if the model is healthy. <---
            if is_better:
                dynamic_pseudo_df.loc[global_i, "pseudo_label"] = int(heldout_preds[local_i])
                dynamic_pseudo_df.loc[global_i, "pseudo_conf"] = float(heldout_probs[local_i])

        if is_better:
            print(f"[Update] Valid improvement! Conf: {current_mean_conf:.4f} | Stability: {stability:.2f} | Class1 Ratio: {pred_ratio:.2f}")
            best_mean_conf = current_mean_conf
        else:
            reason = "Collapsed" if is_collapsed else ("Unstable" if is_unstable else "Low Confidence")
            print(f"[Quarantine] Rejected ({reason}). Retaining previous healthy pseudo-labels to prevent error cascade.")

        # Gold accuracy purely for the report, not used for decisions
        target_metrics = evaluate_full_target(model, df, collator)

        current_labels, current_conf, vote_ratio, vote_count, unanimous_flag = [], [], [], [], []
        for i in range(len(df)):
            v = votes[i]
            p = probs[i]
            vote_count.append(len(v))
            if len(v) == 0:
                current_labels.append(int(df.loc[i, "init_pseudo_label"]))
                current_conf.append(float(df.loc[i, "init_pseudo_conf"]))
                vote_ratio.append(np.nan)
                unanimous_flag.append(False)
                continue
            counts = np.bincount(np.array(v, dtype=int), minlength=2)
            if counts[0] == counts[1]:
                mean0 = np.mean([pp for vv, pp in zip(v, p) if vv == 0]) if 0 in v else 0.0
                mean1 = np.mean([pp for vv, pp in zip(v, p) if vv == 1]) if 1 in v else 0.0
                winner = 1 if mean1 > mean0 else 0
            else:
                winner = int(np.argmax(counts))
            winner_probs = [pp for vv, pp in zip(v, p) if vv == winner]
            winner_conf = float(np.mean(winner_probs)) if len(winner_probs) > 0 else 0.0
            vr = float(counts[winner] / len(v))
            current_labels.append(winner)
            current_conf.append(winner_conf)
            vote_ratio.append(vr)
            unanimous_flag.append(len(set(v)) == 1)

        df["current_pseudo_label"] = np.array(current_labels, dtype=int)
        df["current_pseudo_conf"] = np.array(current_conf, dtype=float)
        df["vote_ratio"] = np.array(vote_ratio, dtype=float)
        df["vote_count"] = np.array(vote_count, dtype=int)
        df["all_votes_same"] = np.array(unanimous_flag, dtype=bool)

        iter_rows.append({
            "task": task_name, "seed": seed, "shots_per_class": GLOBAL_CONFIG["shots_per_class"],
            "n_subsets": GLOBAL_CONFIG["n_subsets"], "stage": f"iter_{iter_id}", "iteration": iter_id,
            "target_loss": target_metrics["loss"], "target_acc": target_metrics["acc"], "target_f1": target_metrics["f1"],
            "subset_size": info["subset_size"], "confident_subset_size": info["confident_subset_size"],
            "refine_train_size": info["refine_train_size"], "invariant_samples": np.nan, "retention_rate": np.nan,
        })

    df = attach_vote_columns(df, votes, probs)
    refine_history_df = pd.concat(refine_histories, axis=0, ignore_index=True) if len(refine_histories) > 0 else pd.DataFrame()

    return model, df, pd.DataFrame(iter_rows), refine_history_df

In [42]:

def run_final_training(task_name: str, seed: int, source_df: pd.DataFrame, voted_target_df: pd.DataFrame, collator, refined_model):
    stage_banner(task_name, "FINAL TRAINING", f"seed={seed}")

    # 1. Apply strict voting rule to get invariant labels
    invariant_target_df, invariant_stats = select_invariant_samples(voted_target_df)

    # 2. Combine highly confident invariant target labels + heavily oversampled source labels
    final_train_pieces = []
    if len(invariant_target_df) > 0:
        inv_df = invariant_target_df[["model_text", "pseudo_label"]].rename(columns={"pseudo_label": "label"})
        final_train_pieces.append(inv_df)

    factor = GLOBAL_CONFIG.get("source_replay_factor_final", 10)
    source_repeated = pd.concat([source_df[["model_text", "label"]]] * factor, ignore_index=True)
    final_train_pieces.append(source_repeated)

    final_train_df = pd.concat(final_train_pieces, ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)

    # 3. Create a FRESH model for final training (Prevents inheriting bad weights from iteration 8)
    dataset_type = TASK_REGISTRY[task_name]["dataset_type"]
    _, _, _, _, final_model = build_model_objects(dataset_type)

    final_loader = make_loader(final_train_df, collator, GLOBAL_CONFIG["batch_size"], True, True)

    # 4. Train final model
    final_model, final_hist_df = train_model(
        model=final_model,
        train_loader=final_loader,
        epochs=GLOBAL_CONFIG["final_epochs"],
        lr=GLOBAL_CONFIG["lr"],
        weight_decay=GLOBAL_CONFIG["weight_decay"],
        l2_alpha=GLOBAL_CONFIG["l2_alpha"],
        task_name=task_name,
        stage_name="FINAL_TRAIN",
        seed=seed,
        iteration=9,
    )

    # 5. Evaluate on full target
    target_eval_df = voted_target_df[["model_text", "gold_label"]].rename(columns={"gold_label": "label"})
    target_metrics = evaluate_full_target(final_model, target_eval_df, collator)

    final_row = {
        "task": task_name,
        "seed": seed,
        "shots_per_class": GLOBAL_CONFIG["shots_per_class"],
        "n_subsets": GLOBAL_CONFIG["n_subsets"],
        "stage": "final",
        "iteration": 9,
        "target_loss": target_metrics["loss"],
        "target_acc": target_metrics["acc"],
        "target_f1": target_metrics["f1"],
        "subset_size": np.nan,
        "confident_subset_size": np.nan,
        "refine_train_size": len(final_train_df),
        "invariant_samples": invariant_stats["invariant_samples"],
        "retention_rate": invariant_stats["retention_rate"],
    }

    print(f"[FINAL RESULT] target_acc={target_metrics['acc']:.4f} target_f1={target_metrics['f1']:.4f}")

    return final_model, final_hist_df, final_row, final_train_df, invariant_target_df, invariant_stats

In [ ]:
def final_pipeline(
    task_name: str,
    seed: Optional[int] = None,
    n_subsets: Optional[int] = None,
    shots_per_class: Optional[int] = None,
):
    if task_name not in TASK_REGISTRY:
        raise ValueError(f"Unknown task_name={task_name}. Available: {list(TASK_REGISTRY.keys())}")

    if seed is not None:
        GLOBAL_CONFIG["seed"] = int(seed)
    if n_subsets is not None:
        GLOBAL_CONFIG["n_subsets"] = int(n_subsets)
    if shots_per_class is not None:
        GLOBAL_CONFIG["shots_per_class"] = int(shots_per_class)

    seed = int(GLOBAL_CONFIG["seed"])
    n_subsets = int(GLOBAL_CONFIG["n_subsets"])
    shots_per_class = int(GLOBAL_CONFIG["shots_per_class"])

    if n_subsets <= 0:
        raise ValueError("n_subsets must be >= 1")
    if shots_per_class <= 0:
        raise ValueError("shots_per_class must be >= 1")

    run_prefix = task_prefix(task_name, seed)

    stage_banner(
        task_name,
        "FINAL PIPELINE START",
        f"seed={seed} shots={shots_per_class} n_subsets={n_subsets}"
    )

    # save exact run config
    run_config = copy.deepcopy(GLOBAL_CONFIG)
    run_config["task_name"] = task_name
    run_config["source_file"] = TASK_REGISTRY[task_name]["source_file"]
    run_config["target_file"] = TASK_REGISTRY[task_name]["target_file"]
    run_config["dataset_type"] = TASK_REGISTRY[task_name]["dataset_type"]

    config_path = os.path.join(GLOBAL_CONFIG["save_configs_dir"], f"{run_prefix}_config.json")
    save_json(run_config, config_path)

    # ---------------------------
    # 1) Source training
    # ---------------------------
    source_out = run_source_training(
        task_name=task_name,
        seed=seed,
        shots_per_class=shots_per_class,
    )

    source_df = source_out["source_df"]
    target_df = source_out["target_df"]
    collator = source_out["collator"]
    model = source_out["model"]
    source_history_df = source_out["source_history_df"]
    report_rows = list(source_out["report_rows"])

    # save source checkpoint immediately
    source_ckpt_path = os.path.join(
        GLOBAL_CONFIG["save_checkpoints_dir"],
        f"{run_prefix}_iter0_source_model.pt"
    )
    torch.save(model.state_dict(), source_ckpt_path)

    # ---------------------------
    # 2) Iterative refinement
    # ---------------------------
    refined_model, voted_target_df, iter_metrics_df, refine_history_df = run_iterative_refinement(
        task_name=task_name,
        seed=seed,
        source_df=source_df,
        target_df=target_df,
        collator=collator,
        model=model,
    )

    if len(iter_metrics_df) > 0:
        report_rows.extend(iter_metrics_df.to_dict("records"))

    refined_ckpt_path = os.path.join(
        GLOBAL_CONFIG["save_checkpoints_dir"],
        f"{run_prefix}_refined_model.pt"
    )
    torch.save(refined_model.state_dict(), refined_ckpt_path)

    # ---------------------------
    # 3) Final training
    # ---------------------------
    final_model, final_hist_df, final_row, final_train_df, invariant_target_df, invariant_stats = run_final_training(
        task_name=task_name,
        seed=seed,
        source_df=source_df,
        voted_target_df=voted_target_df,
        collator=collator,
        refined_model=refined_model,
    )
    report_rows.append(final_row)

    final_ckpt_path = os.path.join(
        GLOBAL_CONFIG["save_checkpoints_dir"],
        f"{run_prefix}_final_model.pt"
    )
    torch.save(final_model.state_dict(), final_ckpt_path)

    # ---------------------------
    # 4) Save reports
    # ---------------------------
    report_df = pd.DataFrame(report_rows)
    report_df = report_df.sort_values("iteration").reset_index(drop=True)

    main_report_path = os.path.join(
        GLOBAL_CONFIG["save_reports_dir"],
        f"{run_prefix}.csv"
    )
    save_csv(report_df, main_report_path)

    source_hist_path = os.path.join(
        GLOBAL_CONFIG["save_histories_dir"],
        f"{run_prefix}_source_history.csv"
    )
    save_csv(source_history_df, source_hist_path)

    refine_hist_path = os.path.join(
        GLOBAL_CONFIG["save_histories_dir"],
        f"{run_prefix}_refine_history.csv"
    )
    if len(refine_history_df) > 0:
        save_csv(refine_history_df, refine_hist_path)
    else:
        pd.DataFrame().to_csv(refine_hist_path, index=False)

    final_hist_path = os.path.join(
        GLOBAL_CONFIG["save_histories_dir"],
        f"{run_prefix}_final_history.csv"
    )
    save_csv(final_hist_df, final_hist_path)

    voted_target_path = os.path.join(
        GLOBAL_CONFIG["save_votes_dir"],
        f"{run_prefix}_voted_target.csv"
    )
    save_csv(voted_target_df, voted_target_path)

    invariant_target_path = os.path.join(
        GLOBAL_CONFIG["save_votes_dir"],
        f"{run_prefix}_invariant_target.csv"
    )
    save_csv(invariant_target_df, invariant_target_path)

    # one-row summary
    summary_df = pd.DataFrame([{
        "task": task_name,
        "seed": seed,
        "shots_per_class": shots_per_class,
        "n_subsets": n_subsets,
        "iter0_target_acc": report_df.loc[report_df["stage"] == "iter_0", "target_acc"].iloc[0] if (report_df["stage"] == "iter_0").any() else np.nan,
        "iter0_target_f1": report_df.loc[report_df["stage"] == "iter_0", "target_f1"].iloc[0] if (report_df["stage"] == "iter_0").any() else np.nan,
        "best_mid_iter_acc": report_df[report_df["stage"].str.startswith("iter_")]["target_acc"].max() if len(report_df) > 0 else np.nan,
        "best_mid_iter_f1": report_df[report_df["stage"].str.startswith("iter_")]["target_f1"].max() if len(report_df) > 0 else np.nan,
        "final_target_acc": final_row["target_acc"],
        "final_target_f1": final_row["target_f1"],
        "invariant_samples": invariant_stats["invariant_samples"],
        "retention_rate": invariant_stats["retention_rate"],
        "invariant_pseudo_acc": invariant_stats["invariant_pseudo_acc"],
        "invariant_pseudo_f1": invariant_stats["invariant_pseudo_f1"],
        "main_report_csv": main_report_path,
        "source_history_csv": source_hist_path,
        "refine_history_csv": refine_hist_path,
        "final_history_csv": final_hist_path,
        "voted_target_csv": voted_target_path,
        "invariant_target_csv": invariant_target_path,
        "source_ckpt": source_ckpt_path,
        "refined_ckpt": refined_ckpt_path,
        "final_ckpt": final_ckpt_path,
        "config_json": config_path,
    }])

    summary_path = os.path.join(
        GLOBAL_CONFIG["save_reports_dir"],
        f"{run_prefix}_summary.csv"
    )
    save_csv(summary_df, summary_path)

    # ---------------------------
    # 5) Console display
    # ---------------------------
    stage_banner(task_name, "RUN COMPLETE", f"seed={seed}")

    print("\nMain per-iteration report:")
    display(report_df)

    print("\nRun summary:")
    display(summary_df)

    print("\nSaved files:")
    print(" main_report_path   :", main_report_path)
    print(" summary_path       :", summary_path)
    print(" source_hist_path   :", source_hist_path)
    print(" refine_hist_path   :", refine_hist_path)
    print(" final_hist_path    :", final_hist_path)
    print(" voted_target_path  :", voted_target_path)
    print(" invariant_target   :", invariant_target_path)
    print(" source_ckpt_path   :", source_ckpt_path)
    print(" refined_ckpt_path  :", refined_ckpt_path)
    print(" final_ckpt_path    :", final_ckpt_path)
    print(" config_path        :", config_path)

    out = {
        "task": task_name,
        "seed": seed,
        "shots_per_class": shots_per_class,
        "n_subsets": n_subsets,
        "report_df": report_df,
        "summary_df": summary_df,
        "source_history_df": source_history_df,
        "refine_history_df": refine_history_df,
        "final_history_df": final_hist_df,
        "voted_target_df": voted_target_df,
        "invariant_target_df": invariant_target_df,
        "final_train_df": final_train_df,
        "invariant_stats": invariant_stats,
        "paths": {
            "main_report_csv": main_report_path,
            "summary_csv": summary_path,
            "source_history_csv": source_hist_path,
            "refine_history_csv": refine_hist_path,
            "final_history_csv": final_hist_path,
            "voted_target_csv": voted_target_path,
            "invariant_target_csv": invariant_target_path,
            "source_ckpt": source_ckpt_path,
            "refined_ckpt": refined_ckpt_path,
            "final_ckpt": final_ckpt_path,
            "config_json": config_path,
        },
        "final_model": final_model,
    }

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out

In [44]:
def print_run_settings():
    print("=" * 100)
    print("Current GLOBAL_CONFIG runtime settings")
    print("=" * 100)
    print("seed                :", GLOBAL_CONFIG["seed"])
    print("shots_per_class     :", GLOBAL_CONFIG["shots_per_class"])
    print("n_subsets           :", GLOBAL_CONFIG["n_subsets"])
    print("batch_size          :", GLOBAL_CONFIG["batch_size"])
    print("source_epochs       :", GLOBAL_CONFIG["source_epochs"])
    print("refinement_epochs   :", GLOBAL_CONFIG["refinement_epochs"])
    print("final_epochs        :", GLOBAL_CONFIG["final_epochs"])
    print("lr                  :", GLOBAL_CONFIG["lr"])
    print("dropout             :", GLOBAL_CONFIG["dropout"])
    print("freeze_plm          :", GLOBAL_CONFIG["freeze_plm"])
    print("train_conf_thresh   :", GLOBAL_CONFIG["train_refine_conf_threshold"])
    print("final_conf_thresh   :", GLOBAL_CONFIG["final_invariant_conf_threshold"])
    print("required_vote_ratio :", GLOBAL_CONFIG["required_vote_ratio"])
    print("replay_refine       :", GLOBAL_CONFIG["source_replay_factor_refine"])
    print("replay_final        :", GLOBAL_CONFIG["source_replay_factor_final"])
    print("label_map           :", GLOBAL_CONFIG["label_map"])
    print("=" * 100)

In [45]:
def inspect_and_run(
    task_name: str,
    seed: Optional[int] = None,
    n_subsets: Optional[int] = None,
    shots_per_class: Optional[int] = None,
    inspect_only: bool = False,
):
    if seed is None:
        seed = GLOBAL_CONFIG["seed"]
    if n_subsets is None:
        n_subsets = GLOBAL_CONFIG["n_subsets"]
    if shots_per_class is None:
        shots_per_class = GLOBAL_CONFIG["shots_per_class"]

    print_run_settings()
    inspect_task_data(task_name, shots_per_class=shots_per_class, seed=seed)

    if inspect_only:
        print("\ninspect_only=True, so training was not started.")
        return None

    return final_pipeline(
        task_name=task_name,
        seed=seed,
        n_subsets=n_subsets,
        shots_per_class=shots_per_class,
    )

In [46]:
result = final_pipeline(task_name="M->S", seed=41, n_subsets=10, shots_per_class=25)
clear_gpu_memory()


[TASK=M->S] [FINAL PIPELINE START] DEVICE=cuda:0 GPU_MEM=0.00G/0.00G seed=41 shots=25 n_subsets=10

[TASK=M->S] [SOURCE TRAINING] DEVICE=cuda:0 GPU_MEM=0.00G/0.00G seed=41 shots=25


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SOURCE_TRAIN | task=M->S | seed=41 | iter=0 | epoch=1/25:   0%|          | 0/2 [00:00<?, ?it/s]

[SOURCE_TRAIN] iter=0 epoch=1/25 train_loss=212.980764 train_acc=0.4000


...

[FINAL_TRAIN] iter=9 epoch=2/2 train_loss=204.460772 train_acc=0.9947
[FINAL RESULT] target_acc=0.6220 target_f1=0.7251

[TASK=M->S] [RUN COMPLETE] DEVICE=cuda:0 GPU_MEM=0.88G/6.32G seed=41

Main per-iteration report:


,task,seed,shots_per_class,n_subsets,stage,iteration,target_loss,target_acc,target_f1,subset_size,confident_subset_size,refine_train_size,invariant_samples,retention_rate
0,M->S,41,25,10,iter_0,0,2.054032,0.7610,0.800667,NaN,NaN,NaN,NaN,NaN
1,M->S,41,25,10,iter_1,1,1.668062,0.8240,0.841727,200.0,194.0,394.0,NaN,NaN
2,M->S,41,25,10,iter_2,2,1.246281,0.8700,0.873786,200.0,194.0,394.0,NaN,NaN
3,M->S,41,25,10,iter_3,3,2.722379,0.7460,0.789735,200.0,196.0,396.0,NaN,NaN
4,M->S,41,25,10,iter_4,4,1.625073,0.8580,0.858283,200.0,197.0,397.0,NaN,NaN
5,M->S,41,25,10,iter_5,5,4.794568,0.6420,0.735207,200.0,197.0,397.0,NaN,NaN
6,M->S,41,25,10,iter_6,6,5.524237,0.5975,0.712603,200.0,199.0,399.0,NaN,NaN
7,M->S,41,25,10,iter_7,7,6.180268,0.5580,0.693481,200.0,200.0,400.0,NaN,NaN
8,M->S,41,25,10,iter_8,8,2.762282,0.7940,0.824830,200.0,198.0,398.0,NaN,NaN
9,M->S,41,25,10,iter_9,9,2.553873,0.5745,0.701089,200.0,196.0,396.0,NaN,NaN



Run summary:


,task,seed,shots_per_class,n_subsets,iter0_target_acc,iter0_target_f1,best_mid_iter_acc,best_mid_iter_f1,final_target_acc,final_target_f1,...,main_report_csv,source_history_csv,refine_history_csv,final_history_csv,voted_target_csv,invariant_target_csv,source_ckpt,refined_ckpt,final_ckpt,config_json
0,M->S,41,25,10,0.761,0.800667,0.87,0.873786,0.622,0.725091,...,./seedwise_paper_impl_outputs/reports/M_S_41.csv,./seedwise_paper_impl_outputs/histories/M_S_41...,./seedwise_paper_impl_outputs/histories/M_S_41...,./seedwise_paper_impl_outputs/histories/M_S_41...,./seedwise_paper_impl_outputs/votes/M_S_41_vot...,./seedwise_paper_impl_outputs/votes/M_S_41_inv...,./seedwise_paper_impl_outputs/checkpoints/M_S_...,./seedwise_paper_impl_outputs/checkpoints/M_S_...,./seedwise_paper_impl_outputs/checkpoints/M_S_...,./seedwise_paper_impl_outputs/configs/M_S_41_c...



Saved files:
 main_report_path   : ./seedwise_paper_impl_outputs/reports/M_S_41.csv
 summary_path       : ./seedwise_paper_impl_outputs/reports/M_S_41_summary.csv
 source_hist_path   : ./seedwise_paper_impl_outputs/histories/M_S_41_source_history.csv
 refine_hist_path   : ./seedwise_paper_impl_outputs/histories/M_S_41_refine_history.csv
 final_hist_path    : ./seedwise_paper_impl_outputs/histories/M_S_41_final_history.csv
 voted_target_path  : ./seedwise_paper_impl_outputs/votes/M_S_41_voted_target.csv
 invariant_target   : ./seedwise_paper_impl_outputs/votes/M_S_41_invariant_target.csv
 source_ckpt_path   : ./seedwise_paper_impl_outputs/checkpoints/M_S_41_iter0_source_model.pt
 refined_ckpt_path  : ./seedwise_paper_impl_outputs/checkpoints/M_S_41_refined_model.pt
 final_ckpt_path    : ./seedwise_paper_impl_outputs/checkpoints/M_S_41_final_model.pt
 config_path        : ./seedwise_paper_impl_outputs/configs/M_S_41_config.json
🧹 GPU Cache Cleared. Current Memory Allocated: 482.82 MB


In [47]:
result = final_pipeline(task_name="M->S", seed=42, n_subsets=10, shots_per_class=25)
clear_gpu_memory()


[TASK=M->S] [FINAL PIPELINE START] DEVICE=cuda:0 GPU_MEM=0.47G/0.53G seed=42 shots=25 n_subsets=10

[TASK=M->S] [SOURCE TRAINING] DEVICE=cuda:0 GPU_MEM=0.47G/0.53G seed=42 shots=25


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SOURCE_TRAIN | task=M->S | seed=42 | iter=0 | epoch=1/25:   0%|          | 0/2 [00:00<?, ?it/s]

[SOURCE_TRAIN] iter=0 epoch=1/25 train_loss=212.882702 train_acc=0.4600


...

[Quarantine] Rejected (Collapsed). Retaining previous healthy pseudo-labels to prevent error cascade.

--- [M->S] Refinement iteration 8/10 ---


REFINE_TRAIN | task=M->S | seed=42 | iter=8 | epoch=1/1:   0%|          | 0/13 [00:00<?, ?it/s]

[REFINE_TRAIN] iter=8 epoch=1/1 train_loss=194.880698 train_acc=0.6487


REFINE_PREDICT | M->S | seed=42 | iter=8:   0%|          | 0/57 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
result = final_pipeline(task_name="M->S", seed=1234, n_subsets=10, shots_per_class=25)
clear_gpu_memory()